# 04 — Type Checking and Conversion in Python

## Learning objectives

By the end of this notebook, you should be able to:

- Inspect values with `type()` and check types with `isinstance()`.
- Explain the difference between checking a type and converting a value.
- Convert compatible values using `int()`, `float()`, `str()`, `bool()`, and `complex()`.
- Understand conversions that truncate information or fail.
- Safely convert text entered by a user.
- Validate input and provide useful error messages.
- Recognize when `isinstance()` is preferable to checking an exact type.

**How to study:** Predict each output before running the code. Try changing the inputs and observe what happens. Examples that demonstrate invalid input use `try`/`except`, so the notebook can run from top to bottom.

## 1. Type checking vs. type conversion

These are two different tasks:

- **Type checking** asks, “What type is this value?” or “Is this value an instance of a type?”
- **Type conversion** attempts to produce a value of another type.

`type(value)` reports the object's type. `isinstance(value, int)` checks whether the value is an instance of `int` or a subclass of it.

In [1]:
value = 42

print("Value:", value)
print("Exact type:", type(value))
print("Type name:", type(value).__name__)
print("Is an integer?", isinstance(value, int))
print("Is a string?", isinstance(value, str))

Value: 42
Exact type: <class 'int'>
Type name: int
Is an integer? True
Is a string? False


## 2. Using `type()`

`type()` is useful for learning and debugging. It returns the exact type of an object.

In [2]:
values = [25, 3.5, 2 + 1j, True, "Python", None, [1, 2], (1, 2), {"a": 1}, {1, 2}]

for value in values:
    print(f"{value!r:16} -> {type(value).__name__}")

25               -> int
3.5              -> float
(2+1j)           -> complex
True             -> bool
'Python'         -> str
None             -> NoneType
[1, 2]           -> list
(1, 2)           -> tuple
{'a': 1}         -> dict
{1, 2}           -> set


## 3. Using `isinstance()`

`isinstance(value, SomeType)` returns `True` or `False`. It can also check several acceptable types by passing a tuple of types.

This is often more flexible than `type(value) is SomeType`, because it respects inheritance.

In [3]:
values = [10, 10.5, True, "10"]

for value in values:
    print(
        repr(value),
        "| isinstance int:", isinstance(value, int),
        "| isinstance int or float:", isinstance(value, (int, float))
    )

10 | isinstance int: True | isinstance int or float: True
10.5 | isinstance int: False | isinstance int or float: True
True | isinstance int: True | isinstance int or float: True
'10' | isinstance int: False | isinstance int or float: False


A detail worth remembering: `bool` is a subclass of `int`, so `isinstance(True, int)` is `True`. If you specifically need an actual integer and want to exclude Booleans, you can check the exact type.

In [4]:
print("isinstance(True, int):", isinstance(True, int))
print("type(True) is int:", type(True) is int)
print("isinstance(7, int):", isinstance(7, int))
print("type(7) is int:", type(7) is int)

isinstance(True, int): True
type(True) is int: False
isinstance(7, int): True
type(7) is int: True


## 4. Converting values to integers with `int()`

`int()` can convert integer-like values and suitable strings to integers.

Important behaviours:

- `int("42")` succeeds.
- `int(7.9)` truncates the fractional part toward zero; it does not round to the nearest integer.
- `int("7.9")` fails because the string is not written as an integer literal.
- `int(True)` returns `1` and `int(False)` returns `0`, though this conversion is not always appropriate for application logic.

In [5]:
print(int("42"))
print(int(7.9))
print(int(-7.9))
print(int(True))
print(int(False))

42
7
-7
1
0


In [6]:
# Why int("7.9") fails:
try:
    print(int("7.9"))
except ValueError as error:
    print(type(error).__name__ + ":", error)

ValueError: invalid literal for int() with base 10: '7.9'


## 5. Converting values to floats with `float()`

`float()` accepts numeric values and numeric strings, including scientific notation. The result may be an approximation when the value cannot be represented exactly in binary floating-point.

In [7]:
print(float(5))
print(float("3.75"))
print(float("-2.5"))
print(float("1.2e3"))

5.0
3.75
-2.5
1200.0


In [8]:
try:
    print(float("three point five"))
except ValueError as error:
    print("Conversion failed:", error)

Conversion failed: could not convert string to float: 'three point five'


## 6. Converting values to strings with `str()`

`str()` produces a human-readable string representation. This is useful when building messages or combining text and numeric values.

Remember that conversion to a string does not preserve the original type: `"25"` is text, while `25` is an integer.

In [9]:
age = 20
age_text = str(age)

print(age_text)
print(type(age_text))
print("Age: " + age_text)

20
<class 'str'>
Age: 20


## 7. Converting values to Booleans with `bool()`

`bool()` follows Python's truth-value rules. Common falsy values include:

- `False`
- numeric zero such as `0` and `0.0`
- empty strings such as `""`
- `None`
- empty collections such as `[]`, `()`, `{}`, and `set()`

Most other values are truthy. In particular, non-empty strings such as `"False"` and `"0"` are truthy because they contain characters.

In [10]:
values = [False, True, 0, 1, 0.0, 2.3, "", "False", "0", None, [], [0], {}, {"x": 0}, set()]

for value in values:
    print(f"bool({value!r}) -> {bool(value)}")

bool(False) -> False
bool(True) -> True
bool(0) -> False
bool(1) -> True
bool(0.0) -> False
bool(2.3) -> True
bool('') -> False
bool('False') -> True
bool('0') -> True
bool(None) -> False
bool([]) -> False
bool([0]) -> True
bool({}) -> False
bool({'x': 0}) -> True
bool(set()) -> False


### Important: parsing text such as `"true"` and `"false"`

`bool("False")` is `True`, so do not use `bool()` to parse a user's textual yes/no response. Instead, normalize the text and explicitly recognize allowed values.

In [11]:
def parse_boolean(text):
    normalized = text.strip().lower()

    if normalized in {"true", "yes", "y", "1"}:
        return True
    if normalized in {"false", "no", "n", "0"}:
        return False

    raise ValueError("Enter true/false, yes/no, y/n, or 1/0.")

for answer in [" YES ", "false", "1", "no"]:
    print(repr(answer), "->", parse_boolean(answer))

try:
    parse_boolean("maybe")
except ValueError as error:
    print("Invalid response:", error)

' YES ' -> True
'false' -> False
'1' -> True
'no' -> False
Invalid response: Enter true/false, yes/no, y/n, or 1/0.


## 8. Converting values to complex numbers

`complex()` creates a complex number. You can provide a real number and an optional imaginary part, or use a properly formatted string.

In [12]:
print(complex(3, 4))
print(complex("2+5j"))
print(complex(7))
print(type(complex(3, 4)))

(3+4j)
(2+5j)
(7+0j)
<class 'complex'>


## 9. Conversion can lose information

Some conversions intentionally change the representation or discard information. Choose a conversion based on what your program needs.

In [13]:
number = 9.99
whole_number = int(number)
text_number = str(number)

print("Original:", number, type(number).__name__)
print("After int():", whole_number, type(whole_number).__name__)
print("After str():", text_number, type(text_number).__name__)

Original: 9.99 float
After int(): 9 int
After str(): 9.99 str


For rounding a float, use `round()` when that is the intended behaviour. It differs from `int()`, which truncates toward zero.

In [14]:
values = [3.2, 3.5, 3.8, -3.2, -3.8]

for value in values:
    print(
        value,
        "| int:", int(value),
        "| round:", round(value)
    )

3.2 | int: 3 | round: 3
3.5 | int: 3 | round: 4
3.8 | int: 3 | round: 4
-3.2 | int: -3 | round: -3
-3.8 | int: -3 | round: -4


## 10. Handling conversion errors safely

Invalid conversions commonly raise `ValueError` (for example, parsing `"hello"` as an integer). Passing a value of an unsuitable kind can raise `TypeError`.

Use `try`/`except` when invalid input is a normal possibility. Catch the specific exception you expect, rather than catching every exception without a reason.

In [15]:
samples = ["18", " 27 ", "3.5", "hello", ""]

for sample in samples:
    try:
        number = int(sample)
        print(f"{sample!r} -> {number}")
    except ValueError:
        print(f"{sample!r} is not a valid integer string")

'18' -> 18
' 27 ' -> 27
'3.5' is not a valid integer string
'hello' is not a valid integer string
'' is not a valid integer string


In [16]:
# ValueError vs TypeError
for label, operation in [
    ("ValueError example", lambda: int("hello")),
    ("TypeError example", lambda: int(None)),
]:
    try:
        operation()
    except (ValueError, TypeError) as error:
        print(label, "->", type(error).__name__)

ValueError example -> ValueError
TypeError example -> TypeError


## 11. Build a reusable conversion function

A useful function should make its expected input and failure behaviour clear. This example returns an integer when conversion succeeds and `None` when the input is not a valid integer string.

In a larger program, you might instead raise the error or return a separate success indicator; choose the approach that best fits the caller.

In [17]:
def try_parse_integer(text):
    try:
        return int(text)
    except (ValueError, TypeError):
        return None

for item in ["42", "-7", "4.2", "Python", None]:
    print(repr(item), "->", try_parse_integer(item))

'42' -> 42
'-7' -> -7
'4.2' -> None
'Python' -> None
None -> None


## 12. Validate numeric input

Conversion alone does not guarantee that a value is appropriate for your program. For example, an integer may parse successfully but still be outside the allowed range.

This example separates parsing from validation.

In [18]:
def parse_age(text):
    try:
        age = int(text)
    except ValueError:
        raise ValueError("Age must be a whole number.") from None

    if not 0 <= age <= 120:
        raise ValueError("Age must be between 0 and 120.")

    return age

for sample in ["20", "0", "121", "-1", "twenty"]:
    try:
        print(repr(sample), "->", parse_age(sample))
    except ValueError as error:
        print(repr(sample), "-> invalid:", error)

'20' -> 20
'0' -> 0
'121' -> invalid: Age must be between 0 and 120.
'-1' -> invalid: Age must be between 0 and 120.
'twenty' -> invalid: Age must be a whole number.


## 13. Type checking is not always input validation

A value can have the expected type but still be invalid for your application's rules. For example, `-5` is an integer, but it may not be a valid quantity of items.

Input validation often includes several checks: type or parsing, range, allowed choices, and relationships to other values.

In [19]:
quantity = -5

print("Is integer?", isinstance(quantity, int))
print("Is a non-negative quantity?", quantity >= 0)

if isinstance(quantity, int) and quantity >= 0:
    print("Quantity accepted")
else:
    print("Quantity rejected")

Is integer? True
Is a non-negative quantity? False
Quantity rejected


## 14. Mini practice exercises

Try these before looking at the review examples.

1. Use `type()` to display the types of an integer, float, Boolean, string, and `None`.
2. Use `isinstance()` to check whether `12`, `12.5`, and `True` are instances of `int`.
3. Convert `"250"` to an integer and `"4.75"` to a float.
4. Explain why `int("4.75")` fails.
5. Print the Boolean values of `""`, `" "`, `"False"`, `0`, and `[]`.
6. Write a `parse_boolean()` function that accepts `"yes"` and `"no"` (case-insensitively).
7. Write a function that returns an integer for valid integer text and returns `None` otherwise.
8. Validate that a number entered as text is between 1 and 10 inclusive.
9. Explain the difference between `int(3.9)` and `round(3.9)`.
10. Explain why checking the type of a value does not prove the value is valid for an application.

In [20]:
# Exercises 1–3
for value in [12, 12.5, True, "Python", None]:
    print(repr(value), type(value).__name__)

print(isinstance(12, int))
print(isinstance(12.5, int))
print(isinstance(True, int))

print(int("250"))
print(float("4.75"))

12 int
12.5 float
True bool
'Python' str
None NoneType
True
False
True
250
4.75


In [21]:
# Exercises 5–6
for value in ["", " ", "False", 0, []]:
    print(repr(value), "->", bool(value))

def parse_yes_no(text):
    normalized = text.strip().lower()
    if normalized == "yes":
        return True
    if normalized == "no":
        return False
    raise ValueError("Please enter yes or no.")

print(parse_yes_no(" YES "))
print(parse_yes_no("no"))

'' -> False
' ' -> True
'False' -> True
0 -> False
[] -> False
True
False


In [22]:
# Exercises 7–9
def safe_int(text):
    try:
        return int(text)
    except (ValueError, TypeError):
        return None

print(safe_int("123"))
print(safe_int("12.3"))

def validate_one_to_ten(text):
    number = int(text)
    if not 1 <= number <= 10:
        raise ValueError("Enter a number from 1 to 10.")
    return number

for text in ["1", "7", "10", "11", "abc"]:
    try:
        print(text, "->", validate_one_to_ten(text))
    except ValueError as error:
        print(text, "-> invalid:", error)

print("int(3.9):", int(3.9))
print("round(3.9):", round(3.9))

123
None
1 -> 1
7 -> 7
10 -> 10
11 -> invalid: Enter a number from 1 to 10.
abc -> invalid: invalid literal for int() with base 10: 'abc'
int(3.9): 3
round(3.9): 4


## 15. Key takeaways

- `type()` inspects the exact type; `isinstance()` checks membership in a type hierarchy.
- Type checking and type conversion solve different problems.
- `int()`, `float()`, `str()`, `bool()`, and `complex()` are common conversion tools.
- `int()` truncates floats toward zero; it does not round.
- `bool()` uses truthiness rules, so every non-empty string—including `"False"`—is truthy.
- Invalid conversions may raise `ValueError` or `TypeError`.
- Parse and validate user input separately.
- A valid type does not guarantee that a value meets your application's rules.

## Completion checklist

- [ ] I can inspect values with `type()` and `isinstance()`.
- [ ] I understand `bool` as a subclass of `int`.
- [ ] I can convert common values safely.
- [ ] I understand truncation and rounding differences.
- [ ] I can explain truthiness and parse textual Booleans safely.
- [ ] I can handle conversion errors.
- [ ] I can validate a converted value against application rules.
- [ ] I have completed the practice exercises.

## Next notebook

Continue with **05 — Numeric Types and Precision**, where you will focus on integer arithmetic, floating-point representation, precision, and reliable numeric comparisons.